# Markanvisningar - Markareal-analys

Analys av `000002ON_20260630-110536.xlsx` (SCB): markareal i hektar per kommun, fördelat på ägarkategori (kommun och landsting / kommunalt bostadsföretag), år 2020.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

FILE = "000002ON_20260630-110536.xlsx"
pd.set_option("display.max_rows", 100)

ModuleNotFoundError: No module named 'pandas'

## Råinläsning

Filen har en "stairstep"-struktur: kolumn A innehåller kommunnamn bara på första raden per kommun (övriga rader är tomma), kolumn B innehåller ägarkategori, kolumn C innehåller hektar för 2020. Data ligger på rad 4 t.o.m. 49 (0-indexerat: rader 3-48 efter `header=None`).

In [ ]:
raw = pd.read_excel(FILE, header=None, skiprows=3, nrows=46, usecols="A:C",
                     names=["kommun_raw", "agarkategori", "hektar"])
raw.head(10)

## Städa till tidy dataframe

In [ ]:
df = raw.copy()
df["kommun_raw"] = df["kommun_raw"].ffill()
df[["kommun_kod", "kommun"]] = df["kommun_raw"].str.split(" ", n=1, expand=True)
df = df.drop(columns="kommun_raw")
df = df[["kommun_kod", "kommun", "agarkategori", "hektar"]]
df["hektar"] = df["hektar"].astype(int)
df.head(10)

## Komplettera med fler kommuner

Hämtar Vellinge, Skurup, Ystad, Trelleborg och Stenungsund (2020, samma ägarkategorier) direkt från SCB:s API för tabell [MI0803D](https://www.statistikdatabasen.scb.se/pxweb/sv/ssd/START__MI__MI0803__MI0803D/MarkagandeRegionAgar/) och lägger till i `df`. Svaret cachas lokalt så anropet bara görs en gång.

In [ ]:
import json
import os
import urllib.request

EXTRA_KOMMUNER = {"1233": "Vellinge", "1264": "Skurup", "1286": "Ystad",
                   "1287": "Trelleborg", "1415": "Stenungsund",
                   "1214": "Svalöv", "1285": "Eslöv", "1281": "Lund"}
AGARKAT = {"2": "kommun och landsting", "8": "kommunalt bostadsföretag"}
SCB_URL = "https://api.scb.se/OV0104/v1/doris/sv/ssd/START/MI/MI0803/MI0803D/MarkagandeRegionAgar"
CACHE_FILE = "scb_extra_kommuner.json"

query = {
    "query": [
        {"code": "Region", "selection": {"filter": "item", "values": list(EXTRA_KOMMUNER)}},
        {"code": "Agarkategori", "selection": {"filter": "item", "values": list(AGARKAT)}},
        {"code": "ContentsCode", "selection": {"filter": "item", "values": ["000002ON"]}},
        {"code": "Tid", "selection": {"filter": "item", "values": ["2020"]}},
    ],
    "response": {"format": "json"},
}

if not os.path.exists(CACHE_FILE):
    req = urllib.request.Request(SCB_URL, data=json.dumps(query).encode("utf-8"),
                                  headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as resp:
        with open(CACHE_FILE, "w", encoding="utf-8") as f:
            f.write(resp.read().decode("utf-8"))

scb_data = json.load(open(CACHE_FILE, encoding="utf-8"))
extra_rows = [
    {"kommun_kod": kod, "kommun": EXTRA_KOMMUNER[kod],
     "agarkategori": AGARKAT[agar], "hektar": int(entry["values"][0])}
    for entry in scb_data["data"]
    for kod, agar, _tid in [entry["key"]]
]

df = pd.concat([df, pd.DataFrame(extra_rows)], ignore_index=True)
df = df.sort_values(["kommun_kod", "agarkategori"]).reset_index(drop=True)
df.tail(10)

## Sammanställning per kommun

In [ ]:
pivot = df.pivot_table(index="kommun", columns="agarkategori", values="hektar", aggfunc="sum", fill_value=0)
pivot["totalt"] = pivot.sum(axis=1)
pivot = pivot.sort_values("totalt", ascending=False)
pivot

In [2]:
df.groupby("agarkategori")["hektar"].agg(["sum", "mean", "count"])

NameError: name 'df' is not defined

## Visualisering

In [ ]:
ax = pivot["totalt"].sort_values().plot.barh(figsize=(8, 8), title="Total markareal (hektar) per kommun, 2020")
ax.set_xlabel("Hektar")
plt.tight_layout()
plt.show()

In [ ]:
ax = pivot.drop(columns="totalt").sort_values("kommun och landsting").plot.barh(
    stacked=True, figsize=(8, 8), title="Markareal per kommun och ägarkategori, 2020")
ax.set_xlabel("Hektar")
plt.tight_layout()
plt.show()

## Karta

Kommungränser hämtas från [okfse/sweden-geojson](https://github.com/okfse/sweden-geojson) (öppen data) och cachas lokalt. Fältet `id` i geojson:en motsvarar `kommun_kod`.

In [ ]:
import os
import urllib.request
import geopandas as gpd

GEOJSON_FILE = "sweden_kommuner.geojson"
GEOJSON_URL = "https://raw.githubusercontent.com/okfse/sweden-geojson/master/swedish_municipalities.geojson"
if not os.path.exists(GEOJSON_FILE):
    urllib.request.urlretrieve(GEOJSON_URL, GEOJSON_FILE)

sverige = gpd.read_file(GEOJSON_FILE)
kommun_totals = df.groupby("kommun_kod", as_index=False)["hektar"].sum()
karta = sverige.merge(kommun_totals, left_on="id", right_on="kommun_kod", how="inner")
karta.head()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 8))
karta.plot(column="hektar", cmap="YlOrRd", legend=True, edgecolor="black", linewidth=0.3, ax=ax)
ax.set_title("Total markareal (hektar) per kommun, 2020")
ax.set_axis_off()
plt.show()

## Antal markanvisningar (partiell, ej i huvuddataframen)

SCB för ingen öppen statistik över *antal markanvisningar* (separat från markareal i hektar). Den siffran följs istället av branschkällor (Markanvisning.se/Bostadspolitik.se, Fastighetsvärlden, Byggindustrin), som nästan uteslutande rankar de mest aktiva kommunerna nationellt (Stockholm, Skellefteå, Uppsala m.fl.). Av våra 23 kommuner hittades data endast för **Malmö** och **Göteborg**, och bara för 2022 och 2025 — inget för 2020, 2021, 2023, 2024, och inget för övriga 21 kommuner (flera artiklar låg bakom betalvägg).

Detta läggs därför i en separat, ofullständig tabell snarare än i huvud-`df`, för att inte ge sken av komplett täckning.

In [ ]:
markanvisningar_partiell = pd.DataFrame([
    {"kommun": "Malmö",    "ar": 2022, "antal_markanvisningar": 12, "antal_bostader": pd.NA,
     "kalla": "https://www.fastighetsvarlden.se/notiser/de-fick-flest-markanvisningar-2022/"},
    {"kommun": "Göteborg", "ar": 2022, "antal_markanvisningar": 12, "antal_bostader": pd.NA,
     "kalla": "https://www.fastighetsvarlden.se/notiser/de-fick-flest-markanvisningar-2022/"},
    {"kommun": "Malmö",    "ar": 2025, "antal_markanvisningar": 16, "antal_bostader": 517,
     "kalla": "https://bostadspolitik.se/nagot-hogre-takt-i-markanvisningarna-2025/"},
    {"kommun": "Göteborg", "ar": 2025, "antal_markanvisningar": pd.NA, "antal_bostader": 1090,
     "kalla": "https://bostadspolitik.se/nagot-hogre-takt-i-markanvisningarna-2025/"},
])
markanvisningar_partiell

## Avslutade markanvisningar — test mot kommunernas egna webbplatser

Eftersom ingen nationell källa har data på *avslutade* markanvisningar per kommun testades istället om kommunerna själva publicerar ett register/lista med status (avslutad/pågående) för 2020–2026. Resultat för fyra mindre/medelstora kommuner: inget register med status eller historik hittades hos någon av dem — bara sidor som visar enbart pågående projekt, eller framåtblickande styrdokument (markanvisningsplaner).

In [ ]:
avslutade_markanvisningar_test = pd.DataFrame([
    {"kommun": "Svalöv",     "register_med_status_hittat": False,
     "kommentar": "Lista med 7-8 namngivna projekt, endast 'pågående', ingen historik",
     "kalla": "https://www.svalov.se/bo-bygg--miljo/aktuella-byggprojekt/pagaende-markanvisning"},
    {"kommun": "Skurup",     "register_med_status_hittat": False,
     "kommentar": "Lista med ca 4-6 aktuella/pågående markanvisningar, inget om avslutade",
     "kalla": "https://www.skurup.se/bygga-bo-och-miljo/samhallsutveckling/markanvisningar"},
    {"kommun": "Varberg",    "register_med_status_hittat": False,
     "kommentar": "Markanvisningsplan 2025-2029 (PDF) är framåtblickande, ingen lista över genomförda",
     "kalla": "https://varberg.se/jobb-och-foretagande/mark-och-lokaler/markanvisningar"},
    {"kommun": "Falkenberg", "register_med_status_hittat": False,
     "kommentar": "Enskilda projektsidor/diarieärenden, ingen central status-lista",
     "kalla": "https://kommun.falkenberg.se/bygga-bo-och-miljo/falkenberg-vaxer/byggprojekt"},
])
avslutade_markanvisningar_test

Test mot fyra storstäder (Lund, Helsingborg, Malmö, Göteborg) gav klart bättre träffar än de mindre kommunerna. Malmö har ett genuint skrapbart register med status och beslutsdatum, men det sträcker sig bara till 2023–2025 (inte hela 2020–2026). Göteborg har en interaktiv e-tjänst som troligen döljer fler poster bakom JavaScript-filtrering (ej verifierat fullt ut). Helsingborg har individuella projektsidor under en "tidigare markanvisningar"-rubrik men ingen sammanställd tabell. Lund har bara enstaka exempel nämnda i löpande text.

In [ ]:
avslutade_markanvisningar_storstader = pd.DataFrame([
    {"kommun": "Lund", "register_med_status_hittat": False,
     "kommentar": "Endast 3 enstaka exempel nämnda löpande i text, ingen arkivsida",
     "kalla": "https://lund.se/foretag-och-naringsliv/mark-exploatering-och-lokaler/markanvisning"},
    {"kommun": "Helsingborg", "register_med_status_hittat": "delvis",
     "kommentar": "Sida 'Tidigare markanvisningar' med minst 7-8 projekt som egna undersidor, ingen sammanställd tabell",
     "kalla": "https://foretagare.helsingborg.se/bransch-och-service/mark-lokaler-och-kartor/markanvisning/tidigare-markanvisningar/"},
    {"kommun": "Malmö", "register_med_status_hittat": True,
     "kommentar": "Strukturerad lista, ~30 poster (2025: 11, 2024: 7, 2023: 6) med projektnamn/byggherre/yta/beslutsdatum. Täcker bara 2023-2025, inte 2020-2022",
     "kalla": "https://malmo.se/Stadsutveckling/For-dig-som-bygger-och-utvecklar/Markanvisning/Avslutade-markanvisningar.html"},
    {"kommun": "Göteborg", "register_med_status_hittat": "delvis",
     "kommentar": "E-tjänst med 'Beslutade'-flik, minst 8 synliga poster, troligen fler bakom JavaScript ('Visa alla'), ej fullt verifierat",
     "kalla": "https://goteborg.se/wps/portal/start/goteborg-vaxer/sa-planeras-staden/plan-och-byggprojekt"},
])
avslutade_markanvisningar_storstader

## Malmö: Jämförelseförfarande vs direktanvisning

Malmös register över avslutade markanvisningar ([malmo.se](https://malmo.se/Stadsutveckling/For-dig-som-bygger-och-utvecklar/Markanvisning/Avslutade-markanvisningar.html)) anger tilldelningsmetod per post. Samtliga 29 poster för 2023–2025 (det enda intervall registret täcker) listas nedan.

In [ ]:
MALMO_KALLA = "https://malmo.se/Stadsutveckling/For-dig-som-bygger-och-utvecklar/Markanvisning/Avslutade-markanvisningar.html"

malmo_markanvisningar = pd.DataFrame([
    # 2025
    {"projekt": "Sege Park, Dagrummet 1", "ar": 2025, "metod": "Jämförelseförfarande"},
    {"projekt": "Sege Park, Trädtoppen 2", "ar": 2025, "metod": "Jämförelseförfarande"},
    {"projekt": "Sege Park, Paviljongen 1", "ar": 2025, "metod": "Jämförelseförfarande"},
    {"projekt": "Mosippan 2, Valdemarsro", "ar": 2025, "metod": "Direktanvisning"},
    {"projekt": "Järnåldern 8, Fosie", "ar": 2025, "metod": "Direktanvisning"},
    {"projekt": "Elisedal, Nosgrimman 4", "ar": 2025, "metod": "Direktanvisning"},
    {"projekt": "Hyllie, öster om skolan", "ar": 2025, "metod": "Direktanvisning"},
    {"projekt": "Röjsaxen 2 & Handräfsan 1, Holma", "ar": 2025, "metod": "Jämförelseförfarande"},
    {"projekt": "Handräfsan 3, Holma", "ar": 2025, "metod": "Jämförelseförfarande"},
    {"projekt": "Hamnen 22:164, del av", "ar": 2025, "metod": "Direktanvisning"},
    {"projekt": "Handluckraren 1, Holma", "ar": 2025, "metod": "Direktanvisning"},
    {"projekt": "Universitetsholmen, Amphitrite 1", "ar": 2025, "metod": "Direktanvisning"},
    {"projekt": "Elisedal, Rosengård 173:3, del av", "ar": 2025, "metod": "Direktanvisning"},
    # 2024
    {"projekt": "Stadion, Innerstaden 9:173", "ar": 2024, "metod": "Direktanvisning"},
    {"projekt": "Hyllie, Bunkeflo 6:8 & 7:6, del av", "ar": 2024, "metod": "Direktanvisning"},
    {"projekt": "Elisedal, Galoppen 1", "ar": 2024, "metod": "Direktanvisning"},
    {"projekt": "Hyllie, Bunkeflo 6:8 & 165:61, del av", "ar": 2024, "metod": "Direktanvisning"},
    {"projekt": "Gulmåran 1, del av", "ar": 2024, "metod": "Direktanvisning"},
    {"projekt": "Hyllie, Hyllie 7:4 & Bunkeflo 10:1, del av", "ar": 2024, "metod": "Direktanvisning"},
    {"projekt": "Hyllie, Hyllie 4:2-3 & Bunkeflo 6:8, del av", "ar": 2024, "metod": "Jämförelseförfarande"},
    # 2023
    {"projekt": "Guldstekeln 6, Valdemarsro", "ar": 2023, "metod": "Direktanvisning"},
    {"projekt": "Hyllie 165:61, del av (JöLa)", "ar": 2023, "metod": "Direktanvisning"},
    {"projekt": "Hyllie 165:61, del av (Otto Magnusson)", "ar": 2023, "metod": "Direktanvisning"},
    {"projekt": "Södra Holma", "ar": 2023, "metod": "Direktanvisning"},
    {"projekt": "Hyllie, Badankan 1, del av", "ar": 2023, "metod": "Direktanvisning"},
    {"projekt": "Hyllie, Utställningen 1", "ar": 2023, "metod": "Jämförelseförfarande"},
    {"projekt": "Hyllie, Reklamen 2", "ar": 2023, "metod": "Direktanvisning"},
    {"projekt": "Hyllie, Bunkeflo 6:8, öster om Parken", "ar": 2023, "metod": "Direktanvisning"},
    {"projekt": "Tygelsjö, Spannmålet 18 & 19", "ar": 2023, "metod": "Jämförelseförfarande"},
])
malmo_markanvisningar["kommun"] = "Malmö"
malmo_markanvisningar["kalla"] = MALMO_KALLA
malmo_markanvisningar["metod"].value_counts()

In [ ]:
malmo_metod_per_ar = malmo_markanvisningar.groupby(["ar", "metod"]).size().unstack(fill_value=0)
ax = malmo_metod_per_ar.plot.bar(figsize=(7, 5), title="Malmö: tilldelningsmetod per år (2023-2025)")
ax.set_xlabel("År")
ax.set_ylabel("Antal markanvisningar")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Göteborg: Jämförelseförfarande vs direktanvisning

Göteborgs Stads markanvisningssida saknar metod-taggning och en fullständig historiklista (knappen "Visa alla" kräver en sessionsbaserad sökning som inte gick att replikera). Metod fick istället pusslas ihop manuellt per ärende från tjänsteutlåtanden och nyhetsartiklar — betydligt mindre tillförlitligt än Malmös register.

Av de 8 kända ärendena under "Beslutade":
- **6 bekräftade** som Jämförelseförfarande (tävling/annonserad utvärdering)
- **1 (Låssby i Torslanda)** sannolikt jämförelseförfarande men ej explicit bekräftat — exkluderad nedan
- **1 (Brunnsbo)** visade sig vara en detaljplaneprocess, inte en markanvisning — exkluderad
- **1 (Småhus vid Ängås Platå)** oklar relation till huvudprojektet — exkluderad

Urvalet (n=6) är litet och ensidigt (ingen direktanvisning hittad), vilket sannolikt beror på att Jämförelseförfarande är Göteborgs uttalade huvudprincip snarare än att direktanvisningar inte förekommer — de är bara svårare att hitta i sökningen. Jämförelsen med Malmö nedan ska därför tolkas med stor försiktighet.

In [ ]:
goteborg_markanvisningar = pd.DataFrame([
    {"projekt": "Gamlestaden, kvarter B2", "diarienummer": "EXF-2024-00256", "ar": 2025,
     "metod": "Jämförelseförfarande",
     "kalla": "https://www4.goteborg.se/prod/intraservice/namndhandlingar/samrumportal.nsf/93ec9160f537fa30c12572aa004b6c1a/c74f18d3c41a8f23c1258b1c002b1d19/$FILE/12_EXN240520.pdf"},
    {"projekt": "Järnbrott, Tunnlandsgatan del 2", "diarienummer": "EXF-2023-01726", "ar": 2024,
     "metod": "Jämförelseförfarande",
     "kalla": "https://goteborg.se/wps/portal?uri=gbglnk%3Agbg.page.bb7386fd-1152-47cb-9da4-d06bd7780a77&markanvisning=EXF-2023-01726"},
    {"projekt": "Masthuggskajen, bygglott G5", "diarienummer": "QMDQIEU0269QCP24", "ar": 2024,
     "metod": "Jämförelseförfarande",
     "kalla": "https://www.byggindustrin.se/byggprojekt/stadsplanering/broods-vinner-markanvisning-pa-masthuggskajen/"},
    {"projekt": "Ängås Platå", "diarienummer": "EXF-2023-00950", "ar": 2023,
     "metod": "Jämförelseförfarande",
     "kalla": "https://www.mynewsdesk.com/se/goteborgsstad/pressreleases/social-haallbarhet-i-fokus-naer-aengaas-plataa-markanvisas-3203228"},
    {"projekt": "Guldhedsgatan", "diarienummer": "EXF-2023-00915", "ar": 2023,
     "metod": "Jämförelseförfarande",
     "kalla": "https://www.kaminsky.se/2023/tillsammans-med-wastbygg-ab-och-udda-vann-vi-tavlingen-for-studentlagenheter-och-forskola-i-guldheden-i-centrala-goteborg/"},
])
goteborg_markanvisningar["kommun"] = "Göteborg"
goteborg_markanvisningar["metod"].value_counts()

## Intressanta enskilda fall: vem vann och varför

Research mot nyhetsartiklar, pressmeddelanden och tjänsteutlåtanden för Skurup, Svalöv, Varberg, Helsingborg, Malmö och Göteborg. Kvaliteten varierar kraftigt mellan kommunerna — vissa fall har rik dokumentation om antal tävlande och motivering, andra bara ett vinnarnamn. **Svalöv** gav inget tävlingsfall alls (bara direktanvisningar utan konkurrens). **Varberg** gav inget fullt verifierat tävlingsfall inom 2020–2026, men en kontrovers (Träslövsläge) där en markanvisning sköts upp efter kritik från ~80 hushåll, samt obekräftade ledtrådar (Etikhus i Västerport/Bua Hamnplan).

**Mest detaljerade fallet: Välluv, Helsingborg (2021).** Catena (med Nowaste Logistics) vann ett logistik/life science-område på ca 16 hektar bland **12 inlämnade anbud** — Wihlborgs (förslaget "FLOW") är en namngiven förlorare, vilket är ovanligt konkret dokumentation. Tätt följt av **Oceanön, Helsingborg (2021)**: JM AB vann bland 9 intresseanmälningar (11 förslag), 3 gick till en fördjupad slutomgång, bedömda på genomförbarhet, hållbarhet, innovation och gestaltning — byggs på en konstgjord ö.

In [ ]:
intressanta_fall = pd.DataFrame([
    {"kommun": "Helsingborg", "projekt": "Välluv (verksamhetsområde)", "ar": 2021,
     "antal_tavlande": "12 anbud", "vinnare": "Catena (med Nowaste Logistics)",
     "motivering": "Logistik/life science-område, ca 16 hektar. Namngiven förlorare: Wihlborgs (förslaget 'FLOW')",
     "kalla": "https://foretagare.helsingborg.se/bransch-och-service/mark-lokaler-och-kartor/markanvisning/tidigare-markanvisningar/"},
    {"kommun": "Helsingborg", "projekt": "Oceanön", "ar": 2021,
     "antal_tavlande": "9 anmälningar / 11 förslag, 3 i slutomgång", "vinnare": "JM AB",
     "motivering": "Genomförbarhet, hållbarhet, innovation, gestaltning (tvärsektoriell utvärderingsgrupp). Byggs på konstgjord ö, ~200 lägenheter",
     "kalla": "https://www.byggvarlden.se/sa-ska-oceanon-i-helsingborg-ros-i-hamn"},
    {"kommun": "Helsingborg", "projekt": "Ryktborsten 2, Laröd", "ar": 2021,
     "antal_tavlande": "ej exakt angivet (\"hård konkurrens från många andra bostadsutvecklare\")", "vinnare": "Riksbyggen (med Jaenecke Arkitekter)",
     "motivering": "48 bostäder, hög hållbarhetsprofil (solceller, gröna tak)",
     "kalla": "https://www.riksbyggen.se"},
    {"kommun": "Malmö", "projekt": "Sege Park", "ar": 2024,
     "antal_tavlande": "28 intresseanmälningar, 11 tilldelade", "vinnare": "BoKlok, Botrygg, ByggVesta, Derome, Trianon, Röda Oasen, MKB, Serneke, Sundprojekt, Tallfarm, Wästbygg",
     "motivering": "Boendekoncept och hållbarhetslösningar enligt Sege Parks utvecklingsstrategi, utöver pris",
     "kalla": "https://www.mynewsdesk.com/se/malmo/pressreleases/markanvisningar-foer-bostaeder-i-sege-park-3349145"},
    {"kommun": "Göteborg", "projekt": "Masthuggskajen, bygglott G5", "ar": 2024,
     "antal_tavlande": "ej angivet", "vinnare": "Broods + White arkitekter (\"Sonyas kulturhus\")",
     "motivering": "Kultur i bottenvåning, gestaltningsambition, miljö/klimatkrav. Utsågs till Årets markanvisning 2024 för transparent process",
     "kalla": "https://www.markanvisning.se/nyheter/masthuggskajen-g5-i-goteborg-ar-arets-markanvisning"},
    {"kommun": "Göteborg", "projekt": "Ängås Platå", "ar": 2023,
     "antal_tavlande": "ej angivet", "vinnare": "JM AB, FB Bostad, Göteborgs Egnahems AB",
     "motivering": "Social hållbarhet — motverka riskområdesklassning (inspirerat av Stockholms 'Fokus Skärholmen')",
     "kalla": "https://www.mynewsdesk.com/se/goteborgsstad/pressreleases/social-haallbarhet-i-fokus-naer-aengaas-plataa-markanvisas-3203228"},
    {"kommun": "Göteborg", "projekt": "Guldhedsgatan", "ar": 2023,
     "antal_tavlande": "ej angivet", "vinnare": "Wästbygg + Udda + Kaminsky Arkitektur",
     "motivering": "Studentbostäder + förskola, Svanenmärkt byggnad. Kriterier ej specificerade i källorna",
     "kalla": "https://www.kaminsky.se/2023/tillsammans-med-wastbygg-ab-och-udda-vann-vi-tavlingen-for-studentlagenheter-och-forskola-i-guldheden-i-centrala-goteborg/"},
    {"kommun": "Skurup", "projekt": "Västeräng (Smörbollen 3 + Vallmon 1)", "ar": 2025,
     "antal_tavlande": "ej angivet", "vinnare": "Stjernplan (Smörbollen 3, ~60 lgh), MT Syd (Vallmon 1, ~40 lgh)",
     "motivering": "Ej specificerad i tillgängliga källor",
     "kalla": "https://press.skurup.se/posts/pressreleases/nya-aktorer-pa-vasterang"},
])
intressanta_fall